# P5 Créer et utiliser une base de données immobilière avec SQL sur Databricks

## Introduction

Dans le cadre du projet DATAImmo, l'entreprise immobilière fictive Laplace Immo, réseau national d'agences immobilières, souhaite centraliser les données relatives aux transactions immobilières et foncières réalisées en France afin de mieux comprendre le marché et de faciliter l'analyse des prix de vente des biens immobiliers.

## Mission

Afin de répondre aux besoins du projet, plusieurs actions sont attendues :
    
* enrichir le modèle de base de données existant à l'aide de référentiels géographiques et de données de recensement de la population ;
* concevoir et déployer une base de données sur Databricks ;
* intégrer les données dans la base de données et permettre leur exploitation à travers des requêtes SQL ;
* garantir la conformité des traitements avec la réglementation RGPD.

Pour cela, trois fichiers de données au format XLSX sont mis à disposition : 
* un extrait des données des Demandes de Valeur Foncière (DVF) issu de l'Open Data ;
* les données de recensement de la population fournies par l'Insee ;
* le référentiel géographique français provenant de data.gouv.fr.

## Méthodologie

### Étape 1 : Analyse et modélisation des données

Ce notebook présente les étapes d'exploration, de nettoyage et de transformation des données brutes avant leur intégration dans la base de données.

* analyser les jeux de données et identifier les variables utiles ;
* élaborer un dictionnaire de données ;
* mettre à jour le schéma relationnel ;
* préparer les données avant leur intégration dans la base de données.

### Étape 2 : Implémentation et exploitation de la base de données
* créer et configurer une base de données sur Databricks ;
* importer les données préparées en respectant les types de données, les contraintes d'intégrité et les relations entre les tables ;
* rédiger des requêtes SQL permettant d'analyser le marché immobilier ;
* préparer une présentation structurée incluant les choix de modélisation, la méthodologie de déploiement et les résultats obtenus (requêtes SQL).  

## Librairies

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option(
    'display.float_format',
    '{:.0f}'.format
)

## Dataframes

In [4]:
xls_communes_insee = pd.ExcelFile('../data/raw/donnees_communes.xlsx')
xls_ref_geographique = pd.ExcelFile('../data/raw/fr-esr-referentiel-geographique.xlsx')
xls_valeur_fonciere = pd.ExcelFile('../data/raw/Valeurs-foncières.xlsx')

print(
    xls_communes_insee.sheet_names, 
    xls_ref_geographique.sheet_names,
    xls_valeur_fonciere.sheet_names
    )

['donnees_communes'] ['fr-esr-referentiel-geographique'] ['Feuil1']


In [5]:
df_communes_insee_raw = pd.read_excel('../data/raw/donnees_communes.xlsx', sheet_name=0)
df_ref_geographique_raw = pd.read_excel('../data/raw/fr-esr-referentiel-geographique.xlsx', sheet_name=0)
df_valeur_fonciere_raw = pd.read_excel('../data/raw/Valeurs-foncières.xlsx', sheet_name=0)

____________

## Analyse exploratoire des données

### 1. Fichier Valeurs foncières

#### 1.1 Aperçu des données

In [6]:
print(f'Ce dataset contient :\n{df_valeur_fonciere_raw.shape[0]} enregistrements et {df_valeur_fonciere_raw.shape[1]} variables')

Ce dataset contient :
34169 enregistrements et 46 variables


In [7]:
df_valeur_fonciere_raw.head()

,Code service CH,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain,Nom de l'acquereur
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2020-01-02,Vente,...,2,2,Appartement,NaN,48,3,NaN,NaN,NaN,GUIRAO
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2020-01-02,Vente,...,2,2,Appartement,NaN,40,1,NaN,NaN,NaN,HARNOIS
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2020-01-02,Vente,...,1,2,Appartement,NaN,82,3,NaN,NaN,NaN,ROGIER
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2020-01-02,Vente,...,1,2,Appartement,NaN,27,1,NaN,NaN,NaN,BOCQUIER
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2020-01-02,Vente,...,1,2,Appartement,NaN,47,2,NaN,NaN,NaN,GUILLOSSOU


In [8]:
df_valeur_fonciere_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34169 entries, 0 to 34168
Data columns (total 46 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Code service CH             0 non-null      float64       
 1   Reference document          0 non-null      float64       
 2   1 Articles CGI              0 non-null      float64       
 3   2 Articles CGI              0 non-null      float64       
 4   3 Articles CGI              0 non-null      float64       
 5   4 Articles CGI              0 non-null      float64       
 6   5 Articles CGI              0 non-null      float64       
 7   No disposition              34169 non-null  int64         
 8   Date mutation               34169 non-null  datetime64[ns]
 9   Nature mutation             34169 non-null  object        
 10  Valeur fonciere             34151 non-null  float64       
 11  No voie                     34036 non-null  float64   

In [9]:
round(df_valeur_fonciere_raw.isna().sum()/df_valeur_fonciere_raw.shape[0]*100)

Code service CH              100
Reference document           100
1 Articles CGI               100
2 Articles CGI               100
3 Articles CGI               100
4 Articles CGI               100
5 Articles CGI               100
No disposition                 0
Date mutation                  0
Nature mutation                0
Valeur fonciere                0
No voie                        0
B/T/Q                         94
Code type de voie              0
Type de voie                   3
Code voie                      0
Voie                           0
Code ID commune                0
Code postal                    0
Commune                        0
Code departement               0
Code commune                   0
Préfixe de section            97
Section                        0
No plan                        0
No Volume                    100
1er lot                        0
Surface Carrez du 1er lot      0
2eme lot                     100
Surface Carrez du 2eme lot   100
3eme lot  

In [10]:
df_valeur_fonciere_raw.duplicated().sum()

np.int64(0)

Audit de qualité :

* Beaucoup de variables 100% valeurs manquantes = vides (to drop)
* Vérifier les variables numériques (si changer le type)

Action :

* Définir un subset de travail


#### 1.2 Subset de travail

In [11]:
subset_valeur_fonciere_raw = df_valeur_fonciere_raw.loc[:, 
    (df_valeur_fonciere_raw.isna().sum() / df_valeur_fonciere_raw.shape[0] * 100) <= 90
].copy()

In [12]:
subset_valeur_fonciere_raw.duplicated().sum()

np.int64(0)

* Changer de type des variables Code postal et Code commune (str pour conserver les zéros)
* Vérifier type de la variable No voie (float ne me semble pas adéquat)
* traiter la variable Nom de l'acquereur (RGPD)

Granularité : chaque ligne représente un bien immobilier impliqué dans une mutation

___________

#### 1.3 Table acquereur

In [13]:
df_acquereur = (
    subset_valeur_fonciere_raw[['Nom de l\'acquereur']]
    .drop_duplicates()
    .reset_index(drop=True)
)
df_acquereur['id_acquereur'] = (
    range(1,len(df_acquereur)+1)
)
df_acquereur = (
    df_acquereur[['id_acquereur', 'Nom de l\'acquereur']]
)
df_acquereur.to_csv('../data/tables/acquereurs.csv', index=False)

_________

#### 1.4 Table mutation RGPD

Table de base

In [14]:
df_mutation_rgpd = (
    subset_valeur_fonciere_raw
    .merge(df_acquereur, on='Nom de l\'acquereur')
    .drop(columns='Nom de l\'acquereur')
)

##### 1.4.1 Analyse des variables

###### <h5>**Variables catégorielles**</h5>

In [15]:
df_mutation_rgpd.select_dtypes(include='object').columns

Index(['Nature mutation', 'Type de voie', 'Code voie', 'Voie', 'Commune',
       'Code departement', 'Section', '1er lot', 'Type local'],
      dtype='object')

*Variable Nature mutation*

In [16]:
df_mutation_rgpd['Nature mutation'].value_counts()

Nature mutation
Vente    34169
Name: count, dtype: int64

Toutes les mutations de la base représentent une vente. Cette variable alors ne sera pas utilisée pour éviter la redondance des données.

_____________

*Variable Type de voie*

In [17]:
df_mutation_rgpd['Type de voie'].unique()

array(['RUE', 'BD', 'RTE', 'RES', 'AV', 'PARC', 'ALL', nan, 'PL', 'SQ',
       'CHE', 'IMP', 'QUAI', 'FG', 'CITE', 'PROM', 'CHEM', 'CRS', 'PAS',
       'MAIL', 'TSSE', 'VC', 'CLOS', 'GR', 'TRA', 'LOT', 'CAMI', 'DSC',
       'CHS', 'DOM', 'COUR', 'SEN', 'VLA', 'MRN', 'VGE', 'RPE', 'PTE',
       'HAM', 'MTE', 'CD', 'COR', 'RLE', 'ESPA', 'QUA', 'MAIS', 'VEN',
       'PASS', 'ROC', 'VTE', 'ESP', 'PLA', 'CC', 'FRM', 'BRTL', 'VIL',
       'GPL', 'CASR', 'CHT', 'RPT', 'VOIE', 'VALL', 'VAL', 'COTE', 'CR',
       'REM', 'PTTE', 'N', 'ART', 'ESC', 'VCHE', 'ZAC', 'HLM', 'PTR',
       'CAE', 'ACH', 'PIST', 'PRV', 'GAL', 'PLAN', 'ILOT'], dtype=object)

In [18]:
df_mutation_rgpd['Type de voie'] = (
    df_mutation_rgpd['Type de voie']
    .astype(str)
    .str.strip()
)

In [19]:
df_mutation_rgpd['Type de voie'].str.len().max()

np.int64(4)

___________

*Variable voie*

In [20]:
df_mutation_rgpd['Voie'] = (
    df_mutation_rgpd['Voie']
    .astype(str)
    .str.strip()
)

In [21]:
df_mutation_rgpd['Voie'].str.len().max()

np.int64(26)

_____________

*Variable Commune*

In [22]:
df_mutation_rgpd['Commune'] = (
    df_mutation_rgpd['Commune']
    .str.strip()
)

In [23]:
df_mutation_rgpd['Commune'].str.len().max()

np.int64(30)

_______________

*Variable Code departement*

In [24]:
df_mutation_rgpd['Code departement'].str.len().max()

np.int64(3)

____________

*Variable Type local*

In [25]:
df_mutation_rgpd['Type local'].value_counts()

Type local
Appartement    31378
Maison          2791
Name: count, dtype: int64

In [26]:
df_mutation_rgpd['Type local'].str.len().max()

np.int64(11)

___________

###### <h5>**Variables numériques**</h5>

In [27]:
df_mutation_rgpd.select_dtypes(include='number').columns

Index(['No disposition', 'Valeur fonciere', 'No voie', 'Code type de voie',
       'Code ID commune', 'Code postal', 'Code commune', 'No plan',
       'Surface Carrez du 1er lot', 'Nombre de lots', 'Code type local',
       'Surface reelle bati', 'Nombre pieces principales', 'id_acquereur'],
      dtype='object')

*Variable No voie*

In [28]:
df_mutation_rgpd['No voie'].describe()

count   34036
mean      450
std      1713
min         1
25%         8
50%        22
75%        66
max      9999
Name: No voie, dtype: float64

In [29]:
df_mutation_rgpd['No voie'] = df_mutation_rgpd['No voie'].astype('Int64')

In [30]:
df_mutation_rgpd['No voie'].nlargest(10)

10118    9999
11279    9999
11602    9999
15507    9999
25697    9999
27635    9999
29329    9999
34002    9999
23997    9991
4061     9901
Name: No voie, dtype: Int64

* Cette variable présente des valeurs aberrantes. 

_________

*Variable Code type de voie*

In [31]:
correspondance_type_voie = (
    subset_valeur_fonciere_raw[[
        'Code type de voie',
        'Type de voie']]
        .drop_duplicates()
        .sort_values(
            by='Code type de voie', 
            ascending=True
            )
)
correspondance_type_voie

,Code type de voie,Type de voie
0,0,RUE
6,1,AV
9,2,ALL
3,3,RTE
334,4,CRS
...,...,...
4702,75,PLA
4955,76,CC
19285,77,PIST
5654,78,VIL


_________

*Variable Code ID commune*

In [32]:
df_mutation_rgpd['Code ID commune'].agg(['min','max'])

min       0
max    3214
Name: Code ID commune, dtype: int64

In [33]:
df_mutation_rgpd['Code ID commune'].is_unique

False

In [34]:
correspondance_id_commune = (
    df_mutation_rgpd[[
        'Code ID commune',
        'Code departement',
        'Code commune',
        'Commune']]
        .drop_duplicates()
        .sort_values(
            by='Code ID commune', 
            ascending=True
            )
)
correspondance_id_commune.head()

,Code ID commune,Code departement,Code commune,Commune
6342,0,01,350,SAINT-ETIENNE-DU-BOIS
0,1,01,103,CHEVRY
660,2,01,143,DIVONNE-LES-BAINS
284,3,01,288,PERON
516,4,01,33,VALSERHONE


In [35]:
correspondance_id_commune['Code ID commune'].is_unique

True

* Le Code ID commune est la clé candidate pour faire la relation entre la table commune et table bien

*Variable code commune*

In [36]:
df_mutation_rgpd['Code commune'] = (
    df_mutation_rgpd['Code commune']
    .astype(str)
    .str.strip()
    .str.zfill(3)
)

*Variable code postal*

In [37]:
df_mutation_rgpd['Code postal'] = (
    df_mutation_rgpd['Code postal']
    .astype('Int64')
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

__________

*Variable Code type local*

In [38]:
df_mutation_rgpd['Code type local'].unique()

array([2, 1])

In [39]:
correspondance_type_local = (
    df_mutation_rgpd[[
        'Code type local',
        'Type local']]
        .drop_duplicates()
        .sort_values(
            by='Code type local', 
            ascending=True
            )
)
correspondance_type_local

,Code type local,Type local
9,1,Maison
0,2,Appartement


__________

*Variable Nombre de pieces*

In [40]:
df_mutation_rgpd['Nombre pieces principales'].isna().sum()

np.int64(0)

In [41]:
df_mutation_rgpd['Nombre pieces principales'].agg(['min','max','mean','median'])

min       0
max      11
mean      3
median    3
Name: Nombre pieces principales, dtype: float64

In [42]:
df_mutation_rgpd['Nombre pieces principales'].value_counts(dropna=False).sort_index()

Nombre pieces principales
0        33
1      6800
2     10050
3      9568
4      5534
5      1671
6       369
7        98
8        28
9        13
10        4
11        1
Name: count, dtype: int64

* Nombre de pièces min = 0 dans 33 registres
* à investiguer si valeur réellement nulle, information non renseignée

__________

*Variable Surface carrez*

In [43]:
df_mutation_rgpd['Surface Carrez du 1er lot'].describe()

count   34169
mean       58
std        53
min         0
25%        35
50%        53
75%        72
max      5153
Name: Surface Carrez du 1er lot, dtype: float64

In [44]:
q1 = df_mutation_rgpd['Surface Carrez du 1er lot'].quantile(.25)
q3 = df_mutation_rgpd['Surface Carrez du 1er lot'].quantile(.75)

iqr = q3-q1

min_ecart = q1-1.5*iqr
max_ecart = q3+1.5*iqr

print(f'iqr {iqr}\nmin_ecart {min_ecart}\nmax_ecart {max_ecart}')

iqr 37.71
min_ecart -21.964999999999996
max_ecart 128.875


In [45]:
len(df_mutation_rgpd[df_mutation_rgpd['Surface Carrez du 1er lot'] >= max_ecart].sort_values('Surface Carrez du 1er lot', ascending=False))

884

In [46]:
df_mutation_rgpd.sort_values(by='Surface Carrez du 1er lot').head(10)

,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,Code type de voie,Type de voie,Code voie,Voie,Code ID commune,...,Section,No plan,1er lot,Surface Carrez du 1er lot,Nombre de lots,Code type local,Type local,Surface reelle bati,Nombre pieces principales,id_acquereur
18520,1,2020-04-27,Vente,50740,8,1,AV,374,GEN DE GAULLE,116,...,AL,368,458,0,3,2,Appartement,26,1,4656
19668,1,2020-05-06,Vente,43500,8,0,RUE,930,DE LA SIRENE,1372,...,F,1013,13,0,2,2,Appartement,18,1,1155
21030,1,2020-05-14,Vente,336740,25,0,RUE,4293,LA GRANGE AUX BELLES,3201,...,BP,8,2012,0,2,2,Appartement,14,1,5817
18812,1,2020-04-29,Vente,289900,6,0,RUE,970,DE LA BOURIE ROUGE,1356,...,AV,89,11,1,4,2,Appartement,107,4,2441
28346,1,2020-06-05,Vente,57000,31,0,RUE,550,EMILE ZOLA,3042,...,AH,703,16,1,2,2,Appartement,13,1,6341
19897,1,2020-05-07,Vente,3000,7,0,RUE,9822,VILLEBOIS MAREUIL,3209,...,AO,30,50,1,1,2,Appartement,80,3,2217
7532,1,2020-02-07,Vente,73000,29,0,RUE,2150,SAINT LOUIS,1670,...,CP,379,1,2,3,1,Maison,65,3,5540
22984,1,2020-05-20,Vente,82000,2,0,RUE,210,DU CENTRE,1103,...,AK,267,36,2,2,2,Appartement,45,2,10134
17234,1,2020-04-09,Vente,538,18,0,RUE,2700,DUPONT DES LOGES,965,...,BP,328,13,2,1,2,Appartement,42,3,1173
22380,1,2020-05-19,Vente,116500,16,1,AV,25,DU JURA,13,...,AM,517,420,2,1,2,Appartement,2,0,560


In [47]:
df_mutation_rgpd.sort_values(by='Surface Carrez du 1er lot', ascending=False).head(10)

,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,Code type de voie,Type de voie,Code voie,Voie,Code ID commune,...,Section,No plan,1er lot,Surface Carrez du 1er lot,Nombre de lots,Code type local,Type local,Surface reelle bati,Nombre pieces principales,id_acquereur
7010,1,2020-02-05,Vente,265000,90,0,RUE,2190,DE LA JUSTICE,2832,...,AO,106,2,5153,2,2,Appartement,53,3,4342
10202,1,2020-02-20,Vente,223645,2,0,RUE,8050,DES 6 FR RUELLAN,1010,...,DA,1374,4,4936,1,2,Appartement,52,3,2751
10713,1,2020-02-21,Vente,2500000,3,2,ALL,94,DES COQUELICOTS,3044,...,AB,496,1,2911,1,2,Appartement,35,1,5462
7630,1,2020-02-07,Vente,4098416,12,0,RUE,440,DU PARC DES SPORTS,2260,...,AW,462,1,1484,1,2,Appartement,45,2,3127
2354,1,2020-01-16,Vente,700000,23,15,BD,3959,GAY LUSSAC,319,...,M,85,2,1291,1,2,Appartement,100,3,2128
14352,1,2020-03-11,Vente,154000,34,4,CRS,1900,VICTOR HUGO,409,...,AC,789,77,815,1,2,Appartement,81,3,2909
11615,1,2020-02-27,Vente,1683000,5229,0,RUE,258,DE LA PRIVADIERE,681,...,AK,382,5,742,1,1,Maison,40,2,6231
16356,1,2020-03-26,Vente,7200000,23,15,BD,794,DE BEAUSEJOUR,3208,...,CN,24,11,595,2,2,Appartement,241,8,8862
18172,1,2020-04-23,Vente,95000,5,0,RUE,40,FERNAND BAZIN,69,...,AB,34,3,570,1,1,Maison,110,6,2868
23022,1,2020-05-20,Vente,1608880,18,0,RUE,6200,PAUL BELLAMY,1277,...,EX,116,11,559,1,2,Appartement,348,9,10141


* Présence de quelques valeurs atypiques/extrêmes.
* Ces anomalies semblent liées à des erreurs de qualité de données.

___________

*Variable Surface reelle bati*

In [48]:
df_mutation_rgpd['Surface reelle bati'].describe()

count   34169
mean       57
std        30
min         1
25%        35
50%        53
75%        72
max       379
Name: Surface reelle bati, dtype: float64

* Distribution plus symétrique.
* Valeurs moins écartées.

__________

*Comparaison entre Surface carrez et Surface reelle bati*

In [49]:
df_mutation_rgpd[['Surface Carrez du 1er lot', 'Surface reelle bati']].describe()

,Surface Carrez du 1er lot,Surface reelle bati
count,34169,34169
mean,58,57
std,53,30
min,0,1
25%,35,35
50%,53,53
75%,72,72
max,5153,379


In [50]:
df_mutation_rgpd[df_mutation_rgpd['Surface Carrez du 1er lot'] > 300].sort_values('Surface Carrez du 1er lot', ascending=False)[['Surface Carrez du 1er lot','Surface reelle bati']].head(10)

,Surface Carrez du 1er lot,Surface reelle bati
7010,5153,53
10202,4936,52
10713,2911,35
7630,1484,45
2354,1291,100
14352,815,81
11615,742,40
16356,595,241
18172,570,110
23022,559,348


In [51]:
(df_mutation_rgpd['Surface Carrez du 1er lot'] >df_mutation_rgpd['Surface reelle bati']).value_counts()

True     18368
False    15801
Name: count, dtype: int64

In [52]:
ratio = (
    df_mutation_rgpd['Surface Carrez du 1er lot'] 
    / 
    df_mutation_rgpd['Surface reelle bati']
)
ratio.sort_values(ascending=False).head(10)

7010    97
10202   95
10713   83
7630    33
13545   28
12078   27
28146   23
11615   19
30804   15
5563    13
dtype: float64

In [53]:
df_mutation_rgpd[ratio >= 10][['Surface Carrez du 1er lot','Surface reelle bati']]

,Surface Carrez du 1er lot,Surface reelle bati
2354,1291,100
5563,173,13
7010,5153,53
7630,1484,45
10202,4936,52
10713,2911,35
11615,742,40
12078,510,19
12720,272,25
13545,422,15


In [54]:
round(
    len(df_mutation_rgpd[
            ratio >= 10
            ])
        /len(df_mutation_rgpd)*100,2)

0.04

In [55]:
round(len(
    df_mutation_rgpd[
        (ratio < 10) 
        & 
        (ratio != 0)]
        )/len(df_mutation_rgpd)*100,2)

99.96

* Les deux variables présentent des distributions globalement similaires.
* Une faible proportion des résultats présente des écarts importants entre les deux mesures.
* Les anomalies identifiées concernent ~0,04% des lignes.

Décision pour l'analyse

* La variable Surface reelle bati est privilégiée pour le calcul du prix au m² afin de limiter la sensibilité aux valeurs extrêmes observées.
* Ce choix vise à améliorer la robustesse du calcul sans remettre en cause la cohérence globale de Surface Carrez.

__________

*Variables Valeur fonciere*

In [56]:
(df_mutation_rgpd['Valeur fonciere'] <= 0).value_counts()

Valeur fonciere
False    34169
Name: count, dtype: int64

In [57]:
df_mutation_rgpd['Valeur fonciere'].agg(['min','max','mean','median'])

min          538
max      9000000
mean      252847
median    169000
Name: Valeur fonciere, dtype: float64

In [58]:
df_mutation_rgpd['Valeur fonciere'].isna().sum()

np.int64(18)

In [59]:
df_mutation_rgpd['Valeur fonciere'].describe()

count     34151
mean     252847
std      325259
min         538
25%      104000
50%      169000
75%      285000
max     9000000
Name: Valeur fonciere, dtype: float64

In [60]:
df_mutation_rgpd[['Valeur fonciere','Surface Carrez du 1er lot','Surface reelle bati']].sort_values(by='Valeur fonciere', ascending=False)

,Valeur fonciere,Surface Carrez du 1er lot,Surface reelle bati
30602,9000000,9,10
5260,8600000,64,62
3624,8577713,21,289
7601,7620000,43,42
9987,7600000,253,200
...,...,...,...
29722,NaN,94,94
31141,NaN,69,41
31699,NaN,27,39
32503,NaN,41,40


In [61]:
q1 = df_mutation_rgpd['Valeur fonciere'].quantile(.25)
q3 = df_mutation_rgpd['Valeur fonciere'].quantile(.75)

iqr = q3-q1

ecart_min = q1 - 1.5*iqr
ecart_max = q3 + 1.5*iqr

print(f'IQR: {iqr}\nEcart min: {ecart_min}\nEcart max: {ecart_max}\n')

IQR: 181000.0
Ecart min: -167500.0
Ecart max: 556500.0



____________

###### <h5>**Variables temporelles**</h5>

In [62]:
df_mutation_rgpd['Date mutation'].dtype

dtype('<M8[ns]')

In [63]:
df_mutation_rgpd['Date mutation'].isna().sum()

np.int64(0)

In [64]:
df_mutation_rgpd['Date mutation'].dt.year.value_counts().sort_index()

Date mutation
2020    34169
Name: count, dtype: int64

In [65]:
df_mutation_rgpd['Date mutation'].agg(['min','max'])

min   2020-01-02
max   2020-06-30
Name: Date mutation, dtype: datetime64[ns]

Période de couverture de ce dataset: 
* Début : 02 janvier 2020
* Fin : 30 juin 2020

_____________

###### <h5>**Nouvelle Variables**</h5>

In [67]:
#id_codedep_codecom

df_mutation_rgpd['id_codedep_codecom'] = (
    df_mutation_rgpd['Code departement'] 
    +
    df_mutation_rgpd['Code commune']
)


###### <h5>**Uniformiser les variables**</h5>

In [68]:
#uniformiser les variables pour base de données
df_mutation_rgpd.columns= (
    df_mutation_rgpd.columns
    .str.lower()
    .str.replace(' ','_')
)

df_mutation_rgpd = (
    df_mutation_rgpd
    .rename(
        columns={
            'type_de_voie':'type_voie',
            'code_id_commune':'id_commune',
            'nombre_pieces_principales':'nb_pieces',
            'surface_carrez_du_1er_lot':'surface_carrez',
            'surface_reelle_bati':'surface_reelle'
            })
)

____________

#### 1.5 Table bien

* Entité : bien
* Clé primaire : ID bien
* Clé étrangère : ID commune

Granularité : chaque ligne représente un bien immobilier unique.

In [168]:
df_bien = (
    df_mutation_rgpd[[
        'no_voie',
        'type_voie',
        'voie',
        'id_codedep_codecom',
        'type_local',
        'code_type_local',
        'nb_pieces',
        'surface_carrez',
        'surface_reelle'
        ]]
).copy()

df_bien.head()

,no_voie,type_voie,voie,id_codedep_codecom,type_local,code_type_local,nb_pieces,surface_carrez,surface_reelle
0,347,RUE,DU CHATEAU,01103,Appartement,2,3,48,48
1,4,BD,EDOUARD BAUDOIN,06004,Appartement,2,1,39,40
2,20,RUE,MARCEAU,06088,Appartement,2,3,80,82
3,550,RTE,DES VESPINS RN7,06123,Appartement,2,1,28,27
4,9300,RES,LES ARPEGES BD DES ABA,13005,Appartement,2,2,47,47


In [169]:
df_bien.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34169 entries, 0 to 34168
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   no_voie             34036 non-null  Int64  
 1   type_voie           34169 non-null  object 
 2   voie                34169 non-null  object 
 3   id_codedep_codecom  34169 non-null  object 
 4   type_local          34169 non-null  object 
 5   code_type_local     34169 non-null  int64  
 6   nb_pieces           34169 non-null  int64  
 7   surface_carrez      34169 non-null  float64
 8   surface_reelle      34169 non-null  int64  
dtypes: Int64(1), float64(1), int64(3), object(4)
memory usage: 2.4+ MB


In [170]:
df_bien.isna().sum()

no_voie               133
type_voie               0
voie                    0
id_codedep_codecom      0
type_local              0
code_type_local         0
nb_pieces               0
surface_carrez          0
surface_reelle          0
dtype: int64

In [171]:
df_bien.duplicated().sum()

np.int64(15)

In [172]:
df_bien = df_bien.drop_duplicates()

In [173]:
#création d'un id unique pour chaque bien
df_bien['id_bien'] = (
    df_bien
    .reset_index(drop=True)
    .index + 1
)

________

In [174]:
df_bien = (
    df_bien[[
        'id_bien',
        'no_voie',
        'type_voie',
        'voie',
        'id_codedep_codecom',
        'type_local',
        'code_type_local',
        'nb_pieces',
        'surface_carrez',
        'surface_reelle'
        ]]
)

In [194]:
df_bien.to_csv('../data/cloud/biens_dvf.csv', index=False)

____________

#### 1.6 Table vente

In [177]:
df_vente = df_mutation_rgpd.merge(df_bien)

df_vente = (
    df_vente[[
        'id_bien',
        'date_mutation',
        'valeur_fonciere'
        ]]
).copy()

In [178]:
df_vente.head()

,id_bien,date_mutation,valeur_fonciere
0,1,2020-01-02,165000
1,2,2020-01-02,355680
2,3,2020-01-02,229500
3,4,2020-01-02,125000
4,5,2020-01-02,90000


In [179]:
df_vente.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34169 entries, 0 to 34168
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_bien          34169 non-null  int64         
 1   date_mutation    34169 non-null  datetime64[ns]
 2   valeur_fonciere  34151 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 801.0 KB


In [180]:
df_vente.duplicated().sum()

np.int64(0)

In [181]:
df_vente['id_bien'].is_unique

False

In [183]:
df_vente['id_vente'] = (
    df_vente
    .reset_index(drop=False)
    .index+1
)

In [184]:
df_vente['id_vente'].is_unique

True

In [185]:
df_vente = (
    df_vente [[
        'id_vente',
        'id_bien', 
        'date_mutation', 
        'valeur_fonciere'
    ]]
)

In [195]:
df_vente.to_csv('../data/cloud/ventes_dvf.csv', index=False)

____________

### 2. Fichier Recensement Communes

Ce jeu de données fournit des informations démographiques des communes - recensement/population

Source : INSEE

#### 2.1 Aperçu des données

In [97]:
#structure du dataset
print(f'Ce dataset contient :\n{df_communes_insee_raw.shape[0]} enregistrements et {df_communes_insee_raw.shape[1]} variables')

Ce dataset contient :
34991 enregistrements et 9 variables


In [98]:
#aperçu des données
df_communes_insee_raw.head()

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
0,84,01,2,08,1,L'Abergement-Clémenciat,779,19,798
1,84,01,1,01,2,L'Abergement-de-Varey,256,1,257
2,84,01,1,01,4,Ambérieu-en-Bugey,14134,380,14514
3,84,01,2,22,5,Ambérieux-en-Dombes,1751,25,1776
4,84,01,1,04,6,Ambléon,112,6,118


In [99]:
#vérifier les types
df_communes_insee_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34991 entries, 0 to 34990
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CODREG  34991 non-null  int64  
 1   CODDEP  34991 non-null  object 
 2   CODARR  34990 non-null  float64
 3   CODCAN  34990 non-null  object 
 4   CODCOM  34991 non-null  int64  
 5   COM     34991 non-null  object 
 6   PMUN    34991 non-null  int64  
 7   PCAP    34991 non-null  int64  
 8   PTOT    34991 non-null  int64  
dtypes: float64(1), int64(5), object(3)
memory usage: 2.4+ MB


In [100]:
#vérifier les valeurs manquantes
df_communes_insee_raw.isna().sum()

CODREG    0
CODDEP    0
CODARR    1
CODCAN    1
CODCOM    0
COM       0
PMUN      0
PCAP      0
PTOT      0
dtype: int64

In [188]:
#vérifier les doublons
df_communes_insee_raw.duplicated().sum()

np.int64(0)

In [102]:
#vérifier unicité de code commune/doublons
print(df_communes_insee_raw['CODCOM'].is_unique)
df_communes_insee_raw.duplicated(subset=['CODCOM']).sum()

False


np.int64(34083)

In [103]:
#clé candidate minimale
df_communes_insee_raw.duplicated(subset=['CODDEP', 'CODCOM']).sum()

np.int64(0)

Audit de qualité :

* Vérifier le dtype de CODARR (float dû à une valeur manquante ?)
* Vérifier les valeurs manquantes : CODARR (1) et CODCOM (1)

Clé candidate : (CODDEP, CODCOM) 
* Clé composite minimale identifiant une commune

Granularité : une commune 
* Chaque ligne représente une commune unique identifiée par le couple (CODDEP, CODCOM) avec ses mesures de population associées.


__________

#### 2.2 Analyse des Variables

**Variables catégorielles**

In [104]:
df_communes_insee_raw.select_dtypes(include=object).columns

Index(['CODDEP', 'CODCAN', 'COM'], dtype='object')

*Variable CODDEP (code département)*

In [105]:
#variable coddep
print(df_communes_insee_raw['CODDEP'].nunique())
df_communes_insee_raw['CODDEP'].unique()

100


array(['01', '02', '03', '04', '05', '06', '07', '08', '09', 10, 11, 12,
       13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29,
       '2A', '2B', 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60,
       61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94,
       95, 971, 972, 973, 974], dtype=object)

In [106]:
#uniformiser le dtype
df_communes_insee_raw['CODDEP'] = (
    df_communes_insee_raw['CODDEP']
    .astype(str)
    .str.strip()
)

In [107]:
#vérifier la longueur des caractères 
df_communes_insee_raw['CODDEP'].str.len().agg(['min','max'])

min    2
max    3
Name: CODDEP, dtype: int64

_________

*Variable CODCAN (code canton)*

In [108]:
#variable codcan
print(df_communes_insee_raw['CODCAN'].nunique())
df_communes_insee_raw['CODCAN'].unique()

59


array(['08', '01', 22, '04', 10, 14, 15, 17, '02', 12, 18, 21, '03', 11,
       23, 13, 16, 99, '07', '09', 20, 19, '06', '05', 97, 98, 26, 94, 24,
       25, 95, 96, 27, 28, 29, 89, 88, 86, 32, 30, 31, 33, 85, 87, 84, 83,
       40, 36, 41, 34, 35, 37, 39, 38, '00', '  ', 93, nan, 91, 92],
      dtype=object)

In [109]:
df_communes_insee_raw['CODCAN'] = (
    df_communes_insee_raw['CODCAN']
    .astype(str)
    .str.strip()
    .replace(r'^\s*$', np.nan, regex=True)
    .replace('nan', np.nan)
)

In [110]:
#vérifier localisation valeur manquante
df_communes_insee_raw['CODCAN'].isna().sum()

np.int64(77)

In [111]:
df_communes_insee_raw['CODCAN'].unique()

array(['08', '01', '22', '04', '10', '14', '15', '17', '02', '12', '18',
       '21', '03', '11', '23', '13', '16', '99', '07', '09', '20', '19',
       '06', '05', '97', '98', '26', '94', '24', '25', '95', '96', '27',
       '28', '29', '89', '88', '86', '32', '30', '31', '33', '85', '87',
       '84', '83', '40', '36', '41', '34', '35', '37', '39', '38', '00',
       nan, '93', '91', '92'], dtype=object)

In [112]:
#vérifier la longueur
df_communes_insee_raw['CODCAN'].str.len().agg(['min','max']).astype(int)

min    2
max    2
Name: CODCAN, dtype: int64

* 76 valeurs manquantes maintenues ;
* Conversion de CODCAN en string pour préserver les zéros

_________

*Variable COM (libellé commune)*

In [113]:
#variable COM
df_communes_insee_raw['COM'].dtype

dtype('O')

In [114]:
#suppression des espaces inutiles
df_communes_insee_raw['COM'] = (
    df_communes_insee_raw['COM']
    .str.strip()
    .str.upper()
)

In [115]:
df_communes_insee_raw['COM'].head()

0    L'ABERGEMENT-CLÉMENCIAT
1      L'ABERGEMENT-DE-VAREY
2          AMBÉRIEU-EN-BUGEY
3        AMBÉRIEUX-EN-DOMBES
4                    AMBLÉON
Name: COM, dtype: object

In [116]:
#nombre de nom de comunnes distinctes
df_communes_insee_raw['COM'].nunique()

32731

In [117]:
#longueur maximale
df_communes_insee_raw['COM'].str.len().max()

np.float64(45.0)

_________

**Variables numériques**

In [118]:
df_communes_insee_raw.select_dtypes(include='number').columns

Index(['CODREG', 'CODARR', 'CODCOM', 'PMUN', 'PCAP', 'PTOT'], dtype='object')

*Variable CODREG (code région)*

In [119]:
df_communes_insee_raw['CODREG'] = (
    df_communes_insee_raw['CODREG']
    .astype(str)
    .str.zfill(2)
)

In [120]:
df_communes_insee_raw['CODREG'].unique()

array(['84', '32', '93', '44', '76', '28', '75', '24', '27', '53', '94',
       '52', '11', '01', '02', '03', '04'], dtype=object)

In [121]:
df_communes_insee_raw['CODREG'].str.len().agg(['min','max'])

min    2
max    2
Name: CODREG, dtype: int64

* Conversion de la variable en string pour préserver les zéros
* longueur max 2


*Variable CODARR (code arrondissement)*

In [122]:
df_communes_insee_raw[df_communes_insee_raw['CODARR'].isna()]

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
34903,01,971,NaN,NaN,127,SAINT-MARTIN,32489,592,33081


In [123]:
df_communes_insee_raw['CODARR'].unique()

array([ 2.,  1.,  4.,  3.,  5.,  6.,  7.,  9.,  8., nan])

In [124]:
df_communes_insee_raw['CODARR'] = (
    df_communes_insee_raw['CODARR']
    .astype('Int64')
    .astype(str)
    .str.zfill(2)
    .replace('<NA>', np.nan)
)

In [125]:
df_communes_insee_raw['CODARR'].unique()

array(['02', '01', '04', '03', '05', '06', '07', '09', '08', nan],
      dtype=object)

In [126]:
df_communes_insee_raw['CODARR'].str.len().max().astype(int)

np.int64(2)

___________

*Variable CODCOM (code commune)*

In [127]:
df_communes_insee_raw['CODCOM'] = (
    df_communes_insee_raw['CODCOM']
    .astype(str)
    .str.zfill(3)
)

In [128]:
df_communes_insee_raw['CODCOM'].str.len().max()

np.int64(3)

____________

*Variables PMUN, PCAP, PTOT*

In [129]:
df_communes_insee_raw.describe()

,PMUN,PCAP,PTOT
count,34991,34991,34991
mean,1915,35,1951
std,8679,132,8791
min,0,0,0
25%,198,4,202
50%,457,9,468
75%,1162,24,1189
max,493465,5167,498596


In [130]:
#check cohérence
(df_communes_insee_raw['PMUN'] + df_communes_insee_raw['PCAP'] == df_communes_insee_raw['PTOT']).value_counts()

True    34991
Name: count, dtype: int64

In [131]:
len(df_communes_insee_raw[df_communes_insee_raw['PTOT'] == 0])

6

In [132]:
df_communes_insee_raw[df_communes_insee_raw['PTOT'] == 0]

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
20069,44,55,03,04,039,BEAUMONT-EN-VERDUNOIS,0,0,0
20080,44,55,03,04,050,BEZONVAUX,0,0,0
20155,44,55,03,04,139,CUMIÈRES-LE-MORT-HOMME,0,0,0
20198,44,55,03,04,189,FLEURY-DEVANT-DOUAUMONT,0,0,0
20235,44,55,03,04,239,HAUMONT-PRÈS-SAMOGNEUX,0,0,0
20294,44,55,03,04,307,LOUVEMONT-CÔTE-DU-POIVRE,0,0,0


#### 2.3 Nouvelle variable (id)

In [203]:
df_communes_insee_raw['CODDEP_CODCOM'] = (
    df_communes_insee_raw['CODDEP']
    +
    df_communes_insee_raw['CODCOM']
)

In [213]:
df_communes_insee_raw['CODDEP_CODCOM'].is_unique

True

__________

#### 2.3 Table Commune

Clé primaire : coddep, codcom (entité : identifie une commune)
Mesures : pmun, pcap, ptot (recensement des populations)


In [212]:
df_population_communes = (
    df_communes_insee_raw[[
            'CODDEP_CODCOM',
            'CODDEP',
            'COM',  
            'PMUN', 
            'PCAP',
            'PTOT'
            ]]
)

df_population_communes.columns = (
    df_population_communes.columns
    .str.lower()
)

df_population_communes = (
    df_population_communes
    .rename(
        columns={
            'coddep_codcom':'id_codedep_codecom',
            'com':'commune'
        }
    )
)

df_population_communes.head()

,id_codedep_codecom,coddep,commune,pmun,pcap,ptot
0,01001,01,L'ABERGEMENT-CLÉMENCIAT,779,19,798
1,01002,01,L'ABERGEMENT-DE-VAREY,256,1,257
2,01004,01,AMBÉRIEU-EN-BUGEY,14134,380,14514
3,01005,01,AMBÉRIEUX-EN-DOMBES,1751,25,1776
4,01006,01,AMBLÉON,112,6,118


In [214]:
df_population_communes.duplicated().sum()

np.int64(0)

In [ ]:
df_population_communes.to_csv('../data/cloud/communes_insee.csv', index=False)

: 

___________

### 3. Fichier Référentiel géographique

Ce jeu de données fournit les différents niveaux d'agrégats géographiques associés aux communes françaises

Source : https://data.enseignementsup-recherche.gouv.fr/explore/dataset/fr-esr-referentiel-geographique/information/?flg=fr-fr

#### 3.1 Aperçu des données

In [136]:
#structure du dataset
print(f'Ce dataset contient :\n{df_ref_geographique_raw.shape[0]} enregistrements et {df_ref_geographique_raw.shape[1]} variables')

Ce dataset contient :
38916 enregistrements et 37 variables


In [137]:
#aperçu des données
df_ref_geographique_raw.head()

,regrgp_nom,reg_nom,reg_nom_old,aca_nom,dep_nom,com_code,com_code1,com_code2,com_id,com_nom_maj_court,...,fd_id,fr_id,fe_id,uu_id_99,au_code,au_id,auc_id,auc_nom,uu_id_10,geolocalisation
0,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01001,1001,1001,C01001,L ABERGEMENT CLEMENCIAT,...,FD111,FR11,FE1,SO,NaN,AU997,C01001,L'Abergement-Clémenciat,SO,"46.1534255214,4.92611354223"
1,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01002,1002,1002,C01002,L ABERGEMENT DE VAREY,...,FD111,FR11,FE1,SO,2,AU002,AU002,Lyon,SO,"46.0091878776,5.42801696363"
2,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01003,1003,1003,C01003,AMAREINS,...,FD111,FR11,FE1,SO,SO,SO,SO,SO,SO,NaN
3,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01004,1004,1004,C01004,AMBERIEU EN BUGEY,...,FD111,FR11,FE1,UU01303,2,AU002,AU002,Lyon,UU01302,"45.9608475114,5.3729257777"
4,Province,Auvergne-Rhône-Alpes,Rhône-Alpes,Lyon,Ain,01005,1005,1005,C01005,AMBERIEUX EN DOMBES,...,FD111,FR11,FE1,SO,2,AU002,AU002,Lyon,SO,"45.9961799872,4.91227250796"


In [138]:
#vérifier les types
df_ref_geographique_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38916 entries, 0 to 38915
Data columns (total 37 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   regrgp_nom         38916 non-null  object
 1   reg_nom            38916 non-null  object
 2   reg_nom_old        38916 non-null  object
 3   aca_nom            38916 non-null  object
 4   dep_nom            38916 non-null  object
 5   com_code           38916 non-null  object
 6   com_code1          38916 non-null  object
 7   com_code2          38916 non-null  object
 8   com_id             38916 non-null  object
 9   com_nom_maj_court  38916 non-null  object
 10  com_nom_maj        38916 non-null  object
 11  com_nom            38916 non-null  object
 12  uu_code            7625 non-null   object
 13  uu_id              38916 non-null  object
 14  uucr_id            38916 non-null  object
 15  uucr_nom           38916 non-null  object
 16  ze_id              38916 non-null  objec

In [139]:
#vérifier le poucentage des valeurs manquantes
round(df_ref_geographique_raw.isna().sum()/df_ref_geographique_raw.shape[0]*100)

regrgp_nom           0
reg_nom              0
reg_nom_old          0
aca_nom              0
dep_nom              0
com_code             0
com_code1            0
com_code2            0
com_id               0
com_nom_maj_court    0
com_nom_maj          0
com_nom              0
uu_code             80
uu_id                0
uucr_id              0
uucr_nom             0
ze_id                0
dep_code             0
dep_id               0
dep_nom_num          0
dep_num_nom          0
aca_code             0
aca_id               0
reg_code             0
reg_id               0
reg_code_old         0
reg_id_old           0
fd_id                0
fr_id                0
fe_id                0
uu_id_99             0
au_code             45
au_id                0
auc_id               0
auc_nom              0
uu_id_10             0
geolocalisation      6
dtype: float64

In [140]:
#vérifier les doublons complets
df_ref_geographique_raw.duplicated().sum()

np.int64(0)

Audit de qualité :

* Beaucoup de variables non nécessaires ou redondantes ;
* variable uu_code 80% de valeurs manquantes ;
* variable geolocalisation 6% de valeur manquantes (à transformer en deux variables = latitude et longitude?) ;
* vérifier pour les variables numériques (si type à changer)

Action :

* Définir un subset pour cette analyse


#### 3.2 Subset de travail

In [141]:
#subset de travail
subset_ref_geographique_raw = (
    df_ref_geographique_raw[
        [
            'regrgp_nom', 
            'reg_code',  
            'reg_nom',  
            'dep_code',
            'dep_nom',
            'com_code',
            'com_nom_maj'
            ]
    ]
).copy()
subset_ref_geographique_raw.head()

,regrgp_nom,reg_code,reg_nom,dep_code,dep_nom,com_code,com_nom_maj
0,Province,84,Auvergne-Rhône-Alpes,1,Ain,01001,L'ABERGEMENT-CLEMENCIAT
1,Province,84,Auvergne-Rhône-Alpes,1,Ain,01002,L'ABERGEMENT-DE-VAREY
2,Province,84,Auvergne-Rhône-Alpes,1,Ain,01003,AMAREINS
3,Province,84,Auvergne-Rhône-Alpes,1,Ain,01004,AMBERIEU-EN-BUGEY
4,Province,84,Auvergne-Rhône-Alpes,1,Ain,01005,AMBERIEUX-EN-DOMBES


In [142]:
#vérifier les types
subset_ref_geographique_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38916 entries, 0 to 38915
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   regrgp_nom   38916 non-null  object
 1   reg_code     38916 non-null  int64 
 2   reg_nom      38916 non-null  object
 3   dep_code     38916 non-null  object
 4   dep_nom      38916 non-null  object
 5   com_code     38916 non-null  object
 6   com_nom_maj  38916 non-null  object
dtypes: int64(1), object(6)
memory usage: 2.1+ MB


#### 3.3 Analyse des variables

*Variable regrgp_nom (libellé de région)*

In [143]:
subset_ref_geographique_raw['regrgp_nom'].unique()

array(['Province', 'Ile-de-France', 'DROM-COM'], dtype=object)

_________

*Variable reg_nom (libellé de région)*

In [144]:
print(subset_ref_geographique_raw['reg_nom'].nunique())
subset_ref_geographique_raw['reg_nom'].unique()

19


array(['Auvergne-Rhône-Alpes', 'Hauts-de-France',
       "Provence-Alpes-Côte d'Azur", 'Grand Est', 'Occitanie',
       'Normandie', 'Nouvelle-Aquitaine', 'Centre-Val de Loire',
       'Bourgogne-Franche-Comté', 'Bretagne', 'Pays de la Loire',
       'Ile-de-France', 'Guadeloupe', 'Martinique', 'Guyane',
       'La Réunion', "Collectivités d'outre-mer", 'Mayotte', 'Corse'],
      dtype=object)

* La France compte officiellement avec 18 régions : collectivités d'outre-mer ne figurent pas sur la liste, bien que présente dans notre dataset.

___________

*Variable dep_code (code département)*

In [145]:
subset_ref_geographique_raw['dep_code'].unique()

array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
       38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
       55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
       72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88,
       89, 90, 91, 92, 93, 94, 95, 971, 972, 973, 974, 975, 976, 977, 978,
       984, 986, 987, 988, 989, '2A', '2B'], dtype=object)

In [146]:
subset_ref_geographique_raw['dep_code'] = (
    subset_ref_geographique_raw['dep_code']
    .astype(str)
    .str.strip()
    .str.zfill(2)
)
subset_ref_geographique_raw['dep_code'].unique()

array(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11',
       '12', '13', '14', '15', '16', '17', '18', '19', '21', '22', '23',
       '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34',
       '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45',
       '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56',
       '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67',
       '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78',
       '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89',
       '90', '91', '92', '93', '94', '95', '971', '972', '973', '974',
       '975', '976', '977', '978', '984', '986', '987', '988', '989',
       '2A', '2B'], dtype=object)

In [147]:
subset_ref_geographique_raw['dep_code'].str.len().agg(['min','max'])

min    2
max    3
Name: dep_code, dtype: int64

________

*Variable dep_nom (libellé du département)*

In [148]:
print(subset_ref_geographique_raw['dep_nom'].nunique())
subset_ref_geographique_raw['dep_nom'].unique()

109


array(['Ain', 'Aisne', 'Allier', 'Alpes-de-Haute-Provence',
       'Hautes-Alpes', 'Alpes-Maritimes', 'Ardèche', 'Ardennes', 'Ariège',
       'Aube', 'Aude', 'Aveyron', 'Bouches-du-Rhône', 'Calvados',
       'Cantal', 'Charente', 'Charente-Maritime', 'Cher', 'Corrèze',
       "Côte-d'Or", "Côtes-d'Armor", 'Creuse', 'Dordogne', 'Doubs',
       'Drôme', 'Eure', 'Eure-et-Loir', 'Finistère', 'Gard',
       'Haute-Garonne', 'Gers', 'Gironde', 'Hérault', 'Ille-et-Vilaine',
       'Indre', 'Indre-et-Loire', 'Isère', 'Jura', 'Landes',
       'Loir-et-Cher', 'Loire', 'Haute-Loire', 'Loire-Atlantique',
       'Loiret', 'Lot', 'Lot-et-Garonne', 'Lozère', 'Maine-et-Loire',
       'Manche', 'Marne', 'Haute-Marne', 'Mayenne', 'Meurthe-et-Moselle',
       'Meuse', 'Morbihan', 'Moselle', 'Nièvre', 'Nord', 'Oise', 'Orne',
       'Pas-de-Calais', 'Puy-de-Dôme', 'Pyrénées-Atlantiques',
       'Hautes-Pyrénées', 'Pyrénées-Orientales', 'Bas-Rhin', 'Haut-Rhin',
       'Rhône', 'Haute-Saône', 'Saône-et-Loire

* La France compte officiellement avec 101 départements : 96 france metropole et 5 outre-mer ('Guadeloupe','Martinique', 'Guyane', 'La Réunion','Mayotte')
* Dans cette liste s'affiche des territoires français qui ne sont pas des "départements" mais possèdent numérotation spéfifique (Collectivités d'Outre-Mer): 'Saint-Pierre-et-Miquelon', 'Saint-Barthélemy', 'Saint-Martin', 'Wallis et Futuna','Polynésie Française', 'Nouvelle-Calédonie','Terres australes et antarctiques françaises' et 'Ile de Clipperton'.

In [149]:
subset_ref_geographique_raw['dep_nom'] = (
    subset_ref_geographique_raw['dep_nom']
    .str.strip()
)
subset_ref_geographique_raw['dep_nom'].str.len().max()

np.int64(43)

________

*Variable com_code (code commune)*

In [150]:
subset_ref_geographique_raw['com_code'].nunique()

38916

In [151]:
subset_ref_geographique_raw['com_code'] = (
    subset_ref_geographique_raw['com_code']
    .astype(str)
    .str.strip()
)
subset_ref_geographique_raw['com_code'].str.len().agg(['min','max'])

min    5
max    6
Name: com_code, dtype: int64

__________

*Variable com_nom_maj (libellé de la commune en majuscule)*

In [152]:
print(subset_ref_geographique_raw['com_nom_maj'].nunique())
subset_ref_geographique_raw['com_nom_maj'].unique()

35525


array(["L'ABERGEMENT-CLEMENCIAT", "L'ABERGEMENT-DE-VAREY", 'AMAREINS',
       ..., 'ZUANI', 'SAN-GAVINO-DI-FIUMORBO', 'CHISA'],
      shape=(35525,), dtype=object)

In [153]:
subset_ref_geographique_raw['com_nom_maj'] = (
    subset_ref_geographique_raw['com_nom_maj']
    .astype(str)
    .str.strip()
)
subset_ref_geographique_raw['com_nom_maj'].str.len().max()

np.int64(32)

______

**Variables numériques**

In [154]:
subset_ref_geographique_raw.select_dtypes(include='number').columns

Index(['reg_code'], dtype='object')

*Variable reg_code (code région)*

In [155]:
subset_ref_geographique_raw['reg_code'].unique()

array([84, 32, 93, 44, 76, 28, 75, 24, 27, 53, 52, 11,  1,  2,  3,  4,  0,
        6, 94])

In [156]:
subset_ref_geographique_raw['reg_code'] = (
    subset_ref_geographique_raw['reg_code']
    .astype(str)
    .str.strip()
    .str.zfill(2)
)
subset_ref_geographique_raw['reg_code'].str.len().agg(['min','max'])

min    2
max    2
Name: reg_code, dtype: int64

__________

#### 3.4 Table Département

In [157]:
df_departement = (
    subset_ref_geographique_raw[
        [
            'reg_code',    
            'dep_code',
            'dep_nom'
            ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
).copy()

df_departement.head()

,reg_code,dep_code,dep_nom
0,84,01,Ain
1,32,02,Aisne
2,84,03,Allier
3,93,04,Alpes-de-Haute-Provence
4,93,05,Hautes-Alpes


In [158]:
df_departement.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   reg_code  109 non-null    object
 1   dep_code  109 non-null    object
 2   dep_nom   109 non-null    object
dtypes: object(3)
memory usage: 2.7+ KB


In [159]:
df_departement = (
    df_departement[[
        'dep_code',
        'dep_nom',
        'reg_code'
    ]]
)

In [196]:
df_departement.to_csv('../data/cloud/departements_fr.csv', index=False)

____________

#### 3.5 Table Région

In [161]:
df_region = (
    subset_ref_geographique_raw[
        [
            'regrgp_nom',
            'reg_code',    
            'reg_nom'
            ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
).copy()

df_region.head()

,regrgp_nom,reg_code,reg_nom
0,Province,84,Auvergne-Rhône-Alpes
1,Province,32,Hauts-de-France
2,Province,93,Provence-Alpes-Côte d'Azur
3,Province,44,Grand Est
4,Province,76,Occitanie


In [162]:
df_region.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   regrgp_nom  19 non-null     object
 1   reg_code    19 non-null     object
 2   reg_nom     19 non-null     object
dtypes: object(3)
memory usage: 588.0+ bytes


In [163]:
df_regions = (
    df_region[[
        'reg_code',
        'reg_nom',
        'regrgp_nom'
    ]]
)

In [197]:
df_region.to_csv('../data/cloud/regions_fr.csv', index=False)

_________

## Validation des relations entre les tables

### 1. Tables Bien et Commune

In [200]:
df_bien['id_codedep_codecom'].str.len().max()

np.int64(6)

In [ ]:
df_bien_communes = (
    df_bien
    .merge(
        df_population_communes, 
        on=['id_codedep_codecom'], 
        how='left'
        )
)

df_bien_communes.head()

,id_bien,no_voie,type_voie,voie,id_codedep_codecom,type_local,code_type_local,nb_pieces,surface_carrez,surface_reelle,coddep,commune,pmun,pcap,ptot
0,1,347,RUE,DU CHATEAU,01103,Appartement,2,3,48,48,01,CHEVRY,2149,47,2196
1,2,4,BD,EDOUARD BAUDOIN,06004,Appartement,2,1,39,40,06,ANTIBES,73438,1085,74523
2,3,20,RUE,MARCEAU,06088,Appartement,2,3,80,82,06,NICE,342669,2859,345528
3,4,550,RTE,DES VESPINS RN7,06123,Appartement,2,1,28,27,06,SAINT-LAURENT-DU-VAR,29169,183,29352
4,5,9300,RES,LES ARPEGES BD DES ABA,13005,Appartement,2,2,47,47,13,AUBAGNE,47535,451,47986
